# Multiple Linear Regression

A learning notebook covering multiple linear regression, feature preparation, model diagnostics, and interpretation.

## Portfolio edition

This notebook was migrated from Danish Ali Syed's Colab learning work. It includes educational starter material and exercises; it is not presented as a production deployment or as wholly original course content.

Saved outputs and execution state have been cleared. Personal Drive links, mounts, and cloud bucket references have been removed. Supply the authorised input data locally; see README.md. The notebook was reviewed statically, not rerun during migration. Results described in the source are historical and have not been independently reproduced.


# Multiple Linear Regression
## Housing Case Study

Problem Statement:

Consider a real estate company that has a dataset containing the prices of properties in the Delhi region. It wishes to use the data to optimise the sale prices of the properties based on important factors such as area, bedrooms, parking, etc.

Essentially, the company wants —

To identify the variables affecting house prices, e.g. area, number of rooms, bathrooms, etc.
To create a linear model that quantitatively relates house prices with variables such as number of rooms, area, number of bathrooms, etc.
To know the accuracy of the model, i.e. how well these variables can predict house prices.

So interpretation is important!


The steps we will follow in this exercise are as follows:
1.   Reading, understanding and visualising the data
2.   Preparing the data for modelling (train-test split, rescaling etc.)
1.   Training the model
2.   Residual analysis
1.   Predictions and evaluation on the test set

In [ ]:
# 1. Reading, understanding and visualising the data

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
# read the data

In [ ]:
from pathlib import Path
BASE_PATH = str(Path("data").resolve())

In [ ]:
housing = pd.read_csv(f"{BASE_PATH}/housing_data.csv")
housing.head()

In [ ]:
housing.shape

In [ ]:
housing.info()

In [ ]:
housing.describe()

In [ ]:
#Visualising contineous variable
sns.pairplot(housing,size=4, aspect=1)
plt.show()

In [ ]:
# Now Visualising the numeric variables
#Visualing the relationship between categorical and contineous variable is box plot
plt.figure(figsize=(10,8))
sns.boxplot(x='mainroad',y='price',data=housing)
plt.show()

In [ ]:
plt.figure(figsize=(20,12))
plt.subplot(2,3,1)
sns.boxplot(x='mainroad',y='price',data=housing)

plt.subplot(2,3,2)
sns.boxplot(x='guestroom',y='price',data=housing)

plt.subplot(2,3,3)
sns.boxplot(x='basement',y='price',data=housing)

plt.subplot(2,3,4)
sns.boxplot(x='hotwaterheating',y='price',data=housing)

plt.subplot(2,3,5)
sns.boxplot(x='airconditioning',y='price',data=housing)

plt.subplot(2,3,6)
sns.boxplot(x='furnishingstatus',y='price',data=housing)

plt.show()

#2. Preparing the data for modelling (train-test split, rescaling etc.)
## Encoding
*   Converting binary vars to 1/0
*   Other catgorical vars to dummy vars

## Splitting into train and test
## Rescaling of variables

In [ ]:
housing.mainroad.value_counts()

In [ ]:
# Yes / No Variables
varlist = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

housing[varlist]

housing[varlist] = housing[varlist].apply(lambda x: x.map({'yes': 1, 'no': 0}))
housing[varlist].head()

In [ ]:
housing.head()

In [ ]:
# Converting Furnishing Status to numerical variable as well and it is done by creating dummy variable
## Dummy Variables
status = pd.get_dummies(housing['furnishingstatus']).astype(int)
status.head()

Now, you don't need three columns. You can drop the furnished column, as the type of furnishing can be identified with just the last two columns where —
*   00 will correspond to furnished
*   01 will correspond to unfurnished
*   10 will correspond to semi-furnished

In [ ]:
# So now we are dropping redundant dummy Variable
status = pd.get_dummies(housing['furnishingstatus'],drop_first=True).astype(int)
status.head()

In [ ]:
#Join the dummy column with main dataset
#Concat the dummy df with the original one

housing = pd.concat([housing,status],axis=1)
housing.head()

In [ ]:
housing = housing.drop('furnishingstatus', axis=1)
housing.head()

In [ ]:
#Split the data into train test
import sklearn
from sklearn.model_selection import train_test_split

In [ ]:
df_train, df_test = train_test_split(housing, train_size=0.7, random_state=100)
print(df_train.shape)
print(df_test.shape)

Rescaling the Features

As you saw in the demonstration for Simple Linear Regression, scaling doesn't impact your model. Here we can see that except for area, all the columns have small integer values. So it is extremely important to rescale the variables so that they have a comparable scale. If we don't have comparable scales, then some of the coefficients as obtained by fitting the regression model might be very large or very small as compared to the other coefficients. This might become very annoying at the time of model evaluation. So it is advised to use standardization or normalization so that the units of the coefficients obtained are all on the same scale. As you know, there are two common ways of rescaling:

*   Min-Max scaling (Normalization): Between 0 and 1
*   Standardisation (mean-0, sigma-1)

This time, we will use MinMax scaling.
![image.png](EMBEDDED_IMAGE_REMOVED)

In [ ]:
#x
# normalization: (x -  xmin)/(xmax -  xmin)   data compressed between 0 and 1
# standardization: (x-mean)/ sigma       ----> (sigma is standard deviation) (the value obtained will have mean=0, sigma=1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
#1.  instantiate an object
scaler = MinMaxScaler()
#Scale only numeric variables not 0/1 (No/Yes) variables, so create a list of only numeric vars
num_vars = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']

#2. Fit on data
      #fit() -> It learns Min, Max (xman, xmin), learns what needs to be done
      #transform() --> Here it computes x-xmin/ xmax-xmin, transforms
      #fit_transform() does both
df_train[num_vars] = scaler.fit_transform(df_train[num_vars])
df_train.head()

In [ ]:
df_train[num_vars].describe()

#3.   Training the model

In [ ]:
#heatmap
plt.figure(figsize=(16, 10))
sns.heatmap(df_train.corr(), annot = True, cmap = "YlGnBu")
plt.show()

In [ ]:
df_train.head()

In [ ]:
# X_train, y_train
y_train = df_train.pop('price') #pop keeps only price
X_train = df_train

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
#Building the model, we will keep on adding one variable at a time and we will look at
#1. Statistical Significance of the variable
#2. VIF co-relation to other variables

#Looking at Proce and Area - since Area is highly corelated
# so
import statsmodels.api as sm

#add a constanct becoz statmodel doesn't have cons by defualt
X_train_sm = sm.add_constant(X_train['area'])

#create first model
lr = sm.OLS(y_train, X_train_sm)

#Fit Model
lr_model = lr.fit()

#see params
lr_model.params

In [ ]:
print(lr_model.summary())

In [ ]:
# add another variable next higher which is bathrooms

X_train_sm = X_train[['area', 'bathrooms']]
X_train_sm = sm.add_constant(X_train_sm)
#create model
lr = sm.OLS(y_train, X_train_sm)

#Fit Model
lr_model = lr.fit()

#see params
#lr_model.summary()
print(lr_model.summary())

In [ ]:
#add bedrooms
# add another variable next higher which is bathrooms

X_train_sm = X_train[['area', 'bathrooms', 'bedrooms']]
X_train_sm = sm.add_constant(X_train_sm)
#create model
lr = sm.OLS(y_train, X_train_sm)

#Fit Model
lr_model = lr.fit()

#see params
#lr_model.summary()
print(lr_model.summary())

One technique (The bottom-up approach) is that we keep on adding the variablt one by one and the 2nd we can use is where we add all the variables first and then remove unwanted variables one by one

In [ ]:
# 2nd option, adding all variables to the model

#looking at all the columns in the data
housing.columns

In [ ]:
#build a model with all variables
X_train_sm = sm.add_constant(X_train)
#create model
lr = sm.OLS(y_train, X_train_sm)
#fit
lr_model = lr.fit()
#params
print(lr_model.summary())

#p_value of 0.05 is good, below 0.05 is low above 0.05 is high. A low p-value indicates that the variable is statistically significant in explaining the dependent variable.

##we could drop some variables analyzing
##Significance (p_values)
##VIF --- The higher the variable is related to other variables the higher the VIF --We usually want VIF of less than 5

VIF of 5 means 5 =1/(1-r2)
                  1-r2 =0.2
                  r2 =0.8 which means almost 80 of variations of this variable is explained by all other variable variations


![image.png](EMBEDDED_IMAGE_REMOVED)

In [ ]:
X_train.head()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

#create a dataframe that will contain the names of all the feature variables and their respected VIF

vif = pd.DataFrame()
vif['Features'] = X_train.columns
vif['VIF'] = [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by = "VIF", ascending = False)
vif

## We could have:
*   High pvalue and high VIF-- Drop
*   High-Low
      *   High p, Low VIF: Remove first
      *   Low p,  High VIF: Remove these in readjusted model after re-analysis   
*   Low p Low VIF

In [ ]:
X = X_train.drop('semi-furnished', axis=1)

In [ ]:
#create another model
#build a model with all variables
X_train_sm = sm.add_constant(X)
#create model
lr = sm.OLS(y_train, X_train_sm)
#fit
lr_model = lr.fit()
#params
print(lr_model.summary())

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF

vif = pd.DataFrame()
vif['Features'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by = "VIF", ascending = False)
vif

In [ ]:
X = X.drop('bedrooms', axis=1)

In [ ]:
#create another model
#build a model with all variables
X_train_sm = sm.add_constant(X)
#create model
lr = sm.OLS(y_train, X_train_sm)
#fit
lr_model = lr.fit()
#params
print(lr_model.summary())

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF

vif = pd.DataFrame()
vif['Features'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by = "VIF", ascending = False)
vif

In [ ]:
X = X.drop('basement', axis=1)

In [ ]:
#create another model
#build a model with all variables
X_train_sm = sm.add_constant(X)
#create model
lr = sm.OLS(y_train, X_train_sm)
#fit
lr_model = lr.fit()
#params
print(lr_model.summary())

We can see that the equation of our best fitted line is:

$ price = 0.236  \times  area + 0.202  \times  bathrooms + 0.11 \times stories + 0.05 \times mainroad + 0.04 \times guestroom + 0.0876 \times hotwaterheating + 0.0682 \times airconditioning + 0.0629 \times parking + 0.0637 \times prefarea - 0.0337 \times unfurnished $


Overall we have a decent model, but we also acknowledge that we could do better.

We have a couple of options:
1. Add new features (bathrooms/bedrooms, area/stories, etc.)
2. Build a non-linear model

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF

vif = pd.DataFrame()
vif['Features'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by = "VIF", ascending = False)
vif

#4. Residual analysis

In [ ]:
#residue = y_train - predicated values of y
y_train_pred = lr_model.predict(X_train_sm)
y_train_pred

In [ ]:
res = y_train - y_train_pred
sns.distplot(res)
#it is centered around 0 and normal distributed

## 5. Predictions and evaluation on the test set

Coefficients in train are done on scaller set, so it is crucial whene ever you use test, you have to re-do the pre-precessing steps to ensure predictions are correct

Fit on data
*   fit() -> It learns Min, Max (xman, xmin), learns what needs to be done >>> Train
*   transform() --> Here it computes x-xmin/ xmax-xmin, transforms >> TEST
*   fit_transform() does both






In [ ]:
#1. create a list of only numeric vars
num_vars = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']

#2. Fit on data
df_test[num_vars] = scaler.transform(df_test[num_vars])
df_test.head()

In [ ]:
df_test.describe()

In [ ]:
y_test = df_test.pop('price')
X_test = df_test

In [ ]:
#add constant
X_test_sm = sm.add_constant(X_test)
X_test_sm.head()

In [ ]:
X_test_sm = X_test_sm.drop(["bedrooms", "semi-furnished", "basement"], axis =1)

In [ ]:
#predictions
y_test_pred = lr_model.predict(X_test_sm)
y_test_pred

In [ ]:
#evaluate
from sklearn.metrics import r2_score, mean_squared_error

r2_score(y_true=y_test, y_pred=y_test_pred)

In [ ]:
# Plotting y_test and y_pred to understand the spread.
fig = plt.figure()
plt.scatter(y_test,y_test_pred)
fig.suptitle('y_test vs y_pred', fontsize=20)              # Plot heading
plt.xlabel('y_test', fontsize=18)                          # X-label
plt.ylabel('y_pred', fontsize=16)                          # Y-label

#Building our Model
This time, we will be using Linear Regression function from SciKit Learn for its compatability with RFE (which is a utility from sklearn)

RFE - Recursive Feature Elimination

In [ ]:
housingRFE = pd.read_csv(f"{BASE_PATH}/housing_data.csv")
housingRFE.head()

In [ ]:
# Defining the map function
varlistRFE =  ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

#Defining the map function
def binary_mapRFE(x):
    return x.map({'yes': 1, "no": 0})

#Applying the function to the housing list

housingRFE[varlistRFE] = housingRFE[varlistRFE].apply(binary_mapRFE)


In [ ]:
housingRFE.head()

In [ ]:
#Dummy Values
statusRFE =pd.get_dummies(housingRFE['furnishingstatus'], drop_first = True).astype(int)
statusRFE.head()

In [ ]:
#Concat to the dataset
housingRFE = pd.concat([housingRFE, statusRFE], axis = 1)
housingRFE.head()

In [ ]:
#Drop 'furnishingstatus'

housingRFE.drop(['furnishingstatus'], axis = 1, inplace = True)
housingRFE.head()

In [ ]:
#Splitting data into Test and Train
df_trainRFE, df_testRFE = train_test_split(housingRFE, train_size = 0.7, test_size= 0.3, random_state= 100)

In [ ]:
#Scaler
num_varsRFE = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']
df_trainRFE[num_varsRFE] = scaler.fit_transform(df_trainRFE[num_varsRFE])
df_trainRFE.head()

In [ ]:
#dividing into X and y sets for the model building
y_trainRFE = df_trainRFE.pop('price')
X_trainRFE = df_trainRFE

In [ ]:
#Importing RFE and Linear Regression
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

In [ ]:
#Running RFE with the output number of the variable equal to 10
lmrfe = LinearRegression()
lmrfe.fit(X_trainRFE, y_trainRFE)


rfe = RFE(lmrfe, n_features_to_select=10)
#running RFE ... n = no. of variables you need e.g 10)
rfe = rfe.fit(X_trainRFE, y_trainRFE)

In [ ]:
#RFE ahs support feature and ranking feature >> Tells if the feature was selected or not and Ranking is for all the features and is ranked in terms of importances

list(zip(X_trainRFE.columns, rfe.support_, rfe.ranking_))

In [ ]:
col = X_trainRFE.columns[rfe.support_] # support is true
col

In [ ]:
X_trainRFE.columns[~rfe.support_] # support is true

In [ ]:
#lets build the model and analyze
X_trainRFET = X_trainRFE[col]
X_trainRFET.head()

In [ ]:
#Adding constant
X_trainRFET_sm = sm.add_constant(X_trainRFET)
lmRFET = sm.OLS(y_trainRFE, X_trainRFET_sm).fit()
print(lmRFET.summary())

# bedrooms has high P value, so not so significant, so we can drop it

In [ ]:
X_trainRFE_new = X_trainRFET_sm.drop(['bedrooms'], axis = 1)

In [ ]:
# Rebuilding the model without bedrooms

In [ ]:
X_trainRFE_new_sm = sm.add_constant(X_trainRFE_new)
lmRFET_new = sm.OLS(y_trainRFE, X_trainRFE_new_sm).fit()
print(lmRFET_new.summary())

In [ ]:
#create a dataframe that will contain the names of all the feature variables and their respected VIF

vif = pd.DataFrame()
X = X_trainRFE_new_sm.drop(['const'], axis=1)
vif['Features'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by = "VIF", ascending = False)
vif

In [ ]:
#residue = y_train - predicated values of y
y_train_pred_RFE = lmRFET_new.predict(X_trainRFE_new_sm)
y_train_pred_RFE

In [ ]:
resRFE = y_trainRFE - y_train_pred_RFE
sns.distplot(resRFE)
#it is centered around 0 and normal distributed

In [ ]:
#Making Predictions
#Apply Scaling

num_varsRFETest = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'price']
df_testRFE[num_varsRFETest] = scaler.transform(df_testRFE[num_varsRFETest])

In [ ]:
y_testRFE = df_testRFE.pop('price')
X_testRFE = df_testRFE

In [ ]:
x_testRFE_new = X_testRFE[X_trainRFE_new.columns.drop('const')]
x_testRFE_new_sm = sm.add_constant(x_testRFE_new)

In [ ]:
y_test_predRFE = lmRFET_new.predict(x_testRFE_new_sm)

In [ ]:
fig =plt.figure()
plt.scatter(y_testRFE, y_test_predRFE)
fig.suptitle('y_test vs y_pred', fontsize=20)
plt.xlabel('y_test', fontsize=18)
plt.ylabel('y_pred', fontsize=16)
#